In [1]:
import swiftlet
from swiftlet import Transformer, Visitor

In [2]:
swiftlet.__version__

'0.1.0'

# Math Expression parser

In [3]:
# Transformer expression
class Solve(Transformer):
    def start(self, children):
        return children[0]

    def expr(self, node):
        if len(node) == 1:
            return node[0]
        if node[1] == '+':
            return node[0] + node[2]
        if node[1] == '-':
            return node[0] - node[2]
        return None

    def number(self, node):
        return node[0]

grammar = """
start: expr
?expr: expr ("+" | "-") number | number
number: DECIMAL | INT
%import (WS, INT, DECIMAL)
%ignore WS
"""

text = "12.34 + 10 - 0.23"

parser = swiftlet.Swiftlet(grammar, debug=True)
parsed = parser.parse(text)


AST of Grammar
start
  rule
    non_terminal  start
    or_expansion
      non_terminal  expr
  rule
    non_terminal  ?expr
    or_expansion
      or_expansion
        expansion
          non_terminal  expr
          or_expansion
            or_expansion
              string  "+"
            string  "-"
          non_terminal  number
      non_terminal  number
  rule
    non_terminal  number
    or_expansion
      or_expansion
        terminal  DECIMAL
      terminal  INT
  import
    name_list
      name_list
        name_list
          terminal  WS
        terminal  INT
      terminal  DECIMAL
  ignore
    or_expansion
      terminal  WS


Terminals
TerminalDef { name: Terminal(WS), value: "[ \\t\\f\\r\\n]+", pattern: PatternRegex(^[ \t\f\r\n]+), max_width: 18446744073709551615 }
TerminalDef { name: Terminal(DECIMAL), value: "\\d+\\.\\d+", pattern: PatternRegex(^\d+\.\d+), max_width: 18446744073709551615 }
TerminalDef { name: Terminal(INT), value: "\\d+", pattern: PatternRegex(^\d+

In [4]:
parsed.pretty_print()

 start
   expr
     expr
       number   12.34
       +
       number   10
     -
     number   0.23


In [5]:
parsed

Tree(start, [Tree(expr, [Tree(expr, [Tree(number, [Token(Type: DECIMAL, Word: "12.34", Start: 0, End: 5, Line: 0)]), +, Tree(number, [Token(Type: INT, Word: "10", Start: 8, End: 10, Line: 0)])]), -, Tree(number, [Token(Type: DECIMAL, Word: "0.23", Start: 13, End: 17, Line: 0)])])])

In [6]:
g = parsed.iter_subtree()

In [7]:
list(g)

[Tree(start, [Tree(expr, [Tree(expr, [Tree(number, [Token(Type: DECIMAL, Word: "12.34", Start: 0, End: 5, Line: 0)]), +, Tree(number, [Token(Type: INT, Word: "10", Start: 8, End: 10, Line: 0)])]), -, Tree(number, [Token(Type: DECIMAL, Word: "0.23", Start: 13, End: 17, Line: 0)])])]),
 Tree(expr, [Tree(expr, [Tree(number, [Token(Type: DECIMAL, Word: "12.34", Start: 0, End: 5, Line: 0)]), +, Tree(number, [Token(Type: INT, Word: "10", Start: 8, End: 10, Line: 0)])]), -, Tree(number, [Token(Type: DECIMAL, Word: "0.23", Start: 13, End: 17, Line: 0)])]),
 Tree(expr, [Tree(number, [Token(Type: DECIMAL, Word: "12.34", Start: 0, End: 5, Line: 0)]), +, Tree(number, [Token(Type: INT, Word: "10", Start: 8, End: 10, Line: 0)])]),
 Tree(number, [Token(Type: DECIMAL, Word: "0.23", Start: 13, End: 17, Line: 0)]),
 Tree(number, [Token(Type: DECIMAL, Word: "12.34", Start: 0, End: 5, Line: 0)]),
 Tree(number, [Token(Type: INT, Word: "10", Start: 8, End: 10, Line: 0)])]

In [8]:
from abc import ABC
from swiftlet import Tree, Token, ExceptionTreeType
class Visitor(ABC):
    def __call__(self, tree: Tree):
        if not isinstance(tree, Tree):
            raise ExceptionTreeType("argument type is not Tree. It's type is {}".format(type(tree)))

        self._visit_tree(tree)

    def _visit_tree(self, tree: Tree):
        if not isinstance(tree, Tree):
            return
        try:
            _user_func = tree.get_name()
            print(f"{_user_func = }")
            getattr(self, _user_func)(tree)
        except AttributeError as e:
            print(e)
        for child in tree.get_children():
            self._visit_tree(child)


In [9]:
class AstVisitor(Visitor):
    def expr(self, tree):
        tree.set_children(["number"])

    def number(self, tree):
        print("-> ", tree)
        tree.set_children(["number"])

visitor = AstVisitor()
parsed.pretty_print()
visitor(parsed)

 start
   expr
     expr
       number   12.34
       +
       number   10
     -
     number   0.23
_user_func = 'start'
'AstVisitor' object has no attribute 'start'
_user_func = 'expr'


In [10]:
parsed.pretty_print()

 start
   expr   number
